In [8]:
def generate_cst_res(pdb_file, target_residues, anchor_res, output):
    """
    target_residues: lista de resíduos a restringir (ex.: [274, 278, 279])
    anchor_res: número do resíduo âncora em cada chain
    """

    coords = {}

    # Ler coordenadas dos CAs
    with open(pdb_file) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                chain = line[21].strip()
                res_num = int(line[22:26].strip())
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])

                coords[(chain, res_num)] = (x, y, z)

    # Descobrir todas as chains presentes
    chains = sorted(set(chain for chain, _ in coords))

    constraints_count = 0

    with open(output, "w") as out:
        for chain in chains:

            if (chain, anchor_res) not in coords:
                print(f"Aviso: chain {chain} não possui o resíduo âncora {anchor_res}.")
                continue

            for res in target_residues:
                if (chain, res) in coords:
                    x, y, z = coords[(chain, res)]

                    out.write(
                        f"CoordinateConstraint CA {res}{chain} "
                        f"CA {anchor_res}{chain} "
                        f"{x:.3f} {y:.3f} {z:.3f} "
                        f"HARMONIC 0.0 1.0\n"
                    )

                    constraints_count += 1
                else:
                    print(f"Aviso: resíduo {res} da chain {chain} não encontrado.")

    print(f"Gerado: {output} com {constraints_count} constraints.")

In [3]:
target_residues = [405, 406, 519, 565, 567, 575, 578, 579, 582, 583, 586, 603, 614, 615, 618]
target_residues_renumber = [numero - 165 for numero in target_residues]

In [4]:
target_residues_renumber

[240, 241, 354, 400, 402, 410, 413, 414, 417, 418, 421, 438, 449, 450, 453]

In [9]:
input_file = "/home/gbiuser/Documents/vitoria/usp-masters/6-preparation/target/dimer/PDE4B_dimer_prerelax.pdb"
# Constrain catalytic site
generate_cst_res(input_file, target_residues_renumber, anchor_res=target_residues_renumber[0], output="cst.cst")

Gerado: cst.cst com 30 constraints.


In [1]:
def renumber_pdb(input_pdb, output_pdb, start=166):
    with open(input_pdb, 'r') as f:
        lines = f.readlines()
    
    current_resnum = None
    new_resnum = start - 1
    
    with open(output_pdb, 'w') as out:
        for line in lines:
            if line.startswith(('ATOM', 'HETATM')):
                res = line[22:26].strip()
                if res != current_resnum:
                    current_resnum = res
                    new_resnum += 1
                line = line[:22] + f'{new_resnum:4d}' + line[26:]
            out.write(line)


In [3]:
order = [
"S_00000071",
"S_00000189",
"S_00000394",
"S_00000455",
"S_00000477",
"S_00000010",
"S_00000054",
"S_00000389",
"S_00000312",
"S_00000303"
]

import os
import glob

abinitio_dir = '/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/'

pdb_files = sorted(glob.glob(os.path.join(abinitio_dir, 'S*.pdb')))

# ordenar arquivos conforme a lista
pdb_files_sorted = sorted(
    pdb_files,
    key=lambda x: order.index(os.path.basename(x).split('.')[0])
)

for i, file in enumerate(pdb_files_sorted):
    print(file, i+1)
    renumber_pdb(
        input_pdb=file,
        output_pdb=abinitio_dir + f"model{i+1}.pdb",
        start=122
    )

/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000071.pdb 1
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000189.pdb 2
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000394.pdb 3
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000455.pdb 4
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000477.pdb 5
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000010.pdb 6
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000054.pdb 7
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000389.pdb 8
/home/gbiuser/Documents/vitoria/usp-masters/data/structure_prediction/abinitio_relax/structs/S_00000312.pdb 9
/home/gbiu

In [ ]:
# ou essa mas o resultado foi igual
import sys

def generate_cst_res(pdb_file, target_residues, anchor_res, output):
    coords = {}
    chains = {}

    with open(pdb_file) as f:
        for line in f:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                res_num = int(line[22:26].strip())
                chain_id = line[21].strip()
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords[res_num] = (x, y, z)
                chains[res_num] = chain_id

    if anchor_res not in coords:
        print(f"Erro: O resíduo âncora {anchor_res} não foi encontrado no PDB.")
        return

    anchor_chain = chains[anchor_res]
    constraints_count = 0
    with open(output, "w") as out:
        for res in target_residues:
            if res in coords:
                x, y, z = coords[res]
                res_chain = chains[res]
                out.write(
                    f"CoordinateConstraint CA {res}{res_chain} CA {anchor_res}{anchor_chain} "
                    f"{x:.3f} {y:.3f} {z:.3f} HARMONIC 0.0 1.0\n"
                )
                constraints_count += 1
            else:
                print(f"Aviso: Resíduo {res} especificado não foi encontrado no PDB e será pulado.")

    print(f"Gerado: {output} com {constraints_count} constraints criadas.")